In [ ]:
from typing import Any

%load_ext autoreload
%autoreload 2

In [ ]:
%run "./dataset.ipynb"

In [ ]:
from dataclasses import dataclass



In [ ]:
def precompute_rope_cache(head_dim: int, max_seq_len : int , base: float = 1e4):
    freqs = 1.0 / (base ** (torch.arange(0, head_dim, 2) / head_dim))
    t = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    cos = torch.cos(freqs)
    sin = torch.sin(freqs)
    return cos, sin
def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    # x shape: [Batch, Seq_len, Num_heads, Head_dim]
    d = x.shape[-1] // 2
    x1, x2 = x[..., :d], x[..., d:]
    rotated = torch.cat((-x2, x1), dim=-1)

    # Match the Head dim and Cut Table Upto Seq Length
    cos = torch.cat([cos, cos], dim=-1)
    sin = torch.cat([sin, sin], dim=-1)
    # Match dim with X
    cos = cos.unsqueeze(0).unsqueeze(2)
    sin = sin.unsqueeze(0).unsqueeze(2)

    return x * cos + rotated * sin

In [ ]:
class CausalSelfAttentionGQA(nn.Module):
    def __init__(self, config : PeakConfig,dropout_layer):
        super().__init__()
        # Learnable Parameters
        self.q_proj = nn.Linear(config.d_model, config.n_heads * config.head_dim, bias=False)
        self.k_proj = nn.Linear(config.d_model, config.n_kv_heads * config.head_dim, bias=False)
        self.v_proj = nn.Linear(config.d_model, config.n_kv_heads * config.head_dim, bias=False)
        self.out_proj = nn.Linear(config.d_model, config.d_model, bias=False)

        self.dropout_layer = dropout_layer
        self.config = config

    def forward(
        self, x: torch.Tensor,rope_cache,attn_mask: torch.Tensor | None = None,kv_cache: tuple[torch.Tensor, torch.Tensor] | None = None
    ) -> tuple[Any, tuple[Tensor, Tensor] | tuple[Tensor, Any] | None]:
        B, S, C = x.shape

        # Linear projections & split into heads
        q = self.q_proj(x).view(B, S, self.config.n_heads, self.config.head_dim)
        k = self.k_proj(x).view(B, S, self.config.n_kv_heads, self.config.head_dim)
        v = self.v_proj(x).view(B, S, self.config.n_kv_heads, self.config.head_dim)


        # Inject RoPE
        if not self.training and kv_cache is not None:
            start_pos = kv_cache[0].size(2)
        else:
            start_pos = 0

        cos,sin = rope_cache
        cos = cos[start_pos:start_pos + S]
        sin = sin[start_pos:start_pos + S]
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        # Reshape to [Batch, Num_heads, Seq_len, Head_dim]
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        # KV Cache(Only for Inference)
        if not self.training and kv_cache is not None:
            past_k, past_v = kv_cache
            # Append current token(s) to previous K/V
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
            new_kv_cache = (k, v)
        elif not self.training:
            new_kv_cache = (k, v)
        else:
            new_kv_cache = None
        # Attention Calculation
        if self.training:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=0.0,is_causal=(attn_mask is None),enable_gqa=True)
        else:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=0.0,is_causal=False,enable_gqa=True)

        out = out.transpose(1, 2).contiguous().view(B, S, C)
        return self.dropout_layer(self.out_proj(out)) , new_kv_cache

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, config : PeakConfig,dropout_layer):
        super().__init__()
        self.gate_proj = nn.Linear(config.d_model, config.d_ff, bias=False)
        self.up_proj = nn.Linear(config.d_model, config.d_ff, bias=False)
        self.down_proj = nn.Linear(config.d_ff, config.d_model, bias=False)
        self.dropout_layer = dropout_layer
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # SwiGLU: (SiLU(x @ W_gate) * (x @ W_up)) @ W_down
        h = self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))
        return self.dropout_layer(h)

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, config : PeakConfig,dropout_layer):
        super().__init__()
        self.norm = nn.RMSNorm(config.d_model,eps=1e-6,elementwise_affine=True)
        self.attn = CausalSelfAttentionGQA(config,dropout_layer)
        self.mlp = SwiGLU(config,dropout_layer)

    def forward(self, x: torch.Tensor, rope_cache : tuple[Tensor,Tensor], attn_mask: torch.Tensor | None = None,kv_cache: tuple[torch.Tensor, torch.Tensor] | None = None
    ) -> tuple[torch.Tensor, torch.Tensor]:
        # Pre-Norm with residual connections
        attn_out, new_kv = self.attn(self.norm(x), rope_cache, attn_mask=attn_mask,kv_cache=kv_cache)
        x = x + attn_out
        x = x + self.mlp(self.norm(x))
        return x, new_kv

In [ ]:
class PeakModel(nn.Module):
    def __init__(self,config : PeakConfig,dropout=0.1,is_inference_mode=False):
        super().__init__()
        # Embedding
        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)
        # Dropout
        if not is_inference_mode:
            self.dropout_layer = nn.Dropout(dropout)
        else:
            self.dropout_layer = nn.Dropout(0.0)
        # Transformer blocks
        self.layers = nn.ModuleList([
            TransformerBlock(config,self.dropout_layer) for _ in range(config.n_layers)
        ])
        # Final RMSNorm
        self.norm = nn.RMSNorm(config.d_model)
        # Output LM Head (Weight-Tied to tok_emb to save parameters)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        # Precompute RoPE tables
        cos,sin = precompute_rope_cache(config.head_dim,config.max_seq_len)
        self.register_buffer("rope_cos",cos,persistent=False)
        self.register_buffer("rope_sin",sin,persistent=False)

        self.is_inference_mode = is_inference_mode

    def forward(self,input_ids: torch.Tensor,attention_mask: torch.Tensor | None = None,labels: torch.Tensor | None = None,kv_cache = None
    ):
        B, S = input_ids.shape
        device = input_ids.device
        # Embed tokens
        x = self.tok_emb(input_ids)
        # ============================================================
        # TRAINING
        # ============================================================
        if not self.is_inference_mode:
            # Combine causal mask with DataLoader padding mask (if provided)
            combined_mask = None
            if attention_mask is not None:
                causal_mask = torch.tril(torch.ones((S, S), dtype=torch.bool, device=device)).view(1, 1, S, S)
                # Reshape padding mask to: [B, 1, 1, S]
                pad_mask = attention_mask[:, None, None, :].bool()
                # Combine: True means keep, False means mask out
                combined_bool_mask = causal_mask & pad_mask
                combined_mask = torch.zeros((B, 1, S, S), dtype=torch.float32, device=device)
                combined_mask = combined_mask.masked_fill(~combined_bool_mask, float("-inf"))
            #------------- Pass through all blocks-----------------
            for layer in self.layers:
                x,_  = layer(x, rope_cache=(self.rope_cos,self.rope_sin), attn_mask=combined_mask)
            #------------- Pass through all blocks-----------------
            # Final norm and linear projection
            x = self.norm(x)
            logits = self.lm_head(x)  # Shape: [B, S, vocab_size]
            # Loss Calculation
            loss = None
            if labels is not None:
                loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
                loss = loss_fn(
                    logits.view(-1, logits.size(-1)),
                    labels.view(-1)
                )
            return logits , loss
        else:

            new_kv_cache = []
            #------------- Pass through all blocks with Cache-----------------
            for i,layer in enumerate(self.layers):

                past_kv = None if kv_cache is None else kv_cache[i]

                x,layer_kv_cache  = layer(x, rope_cache=(self.rope_cos,self.rope_sin), attn_mask=None,kv_cache=past_kv)

                new_kv_cache.append(layer_kv_cache)

            # Final norm and linear projection
            x = self.norm(x)
            logits = self.lm_head(x)  # Shape: [B, S, vocab_size]
            return logits, new_kv_cache


In [ ]:
# config = PeakConfig(vocab_size=10000,max_seq_len=512,d_model=384,n_layers=6,n_heads=6,n_kv_heads=2,d_ff=4 * 384)
# model = PeakModel(config)

In [15]:
for d in train_loader:
    i,a,o = d['input_ids'],d['attention_mask'],d['labels']
    print(i.shape,a.shape,o.shape)
    print(i.dtype,a.dtype,o.dtype)
    # model(i)
    break

torch.Size([32, 240]) torch.Size([32, 240]) torch.Size([32, 240])
torch.uint16 torch.bool torch.int16
